# 🚩 Sprint 1 · Checkpoint 03 — Velocity

**Companion ao Boyd (Cap 3 — Convex Functions, Cap 4 — Convex Problems) + Weinberger (Lec 11 Logistic, Lec 13 Ridge).**

GD puro é o algoritmo mais simples — e mostra-se rapidamente que é **lento** quando a paisagem do erro é alongada (vales estreitos). Este checkpoint adiciona-lhe três upgrades históricos:

- **Momentum** (Polyak, 1964) — lembra-te da direção anterior; uma bola a rolar.
- **Nesterov** (1983) — espreita o futuro antes de saltar.
- **AdaGrad** (Duchi et al., 2011) — passos diferentes para cada parâmetro.

No fim aplicas tudo a um problema de **regressão linear** (least squares) e comparas curvas de convergência.

**Como trabalhar:**
- Cada `# TODO` é uma micro-task. Substitui `...` ou completa o esqueleto.
- A célula corre uma `assert` — se falhar, lê a mensagem e corrige.
- Trabalha em `solutions/local.ipynb` (cópia local).

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

def check(cond, msg='falhou'):
    assert cond, f'❌ {msg}'
    print('✅', msg)

## §1 — O problema: vales alongados

Considera a função quadrática:

$$ f(x, y) = \tfrac{1}{2}(x^2 + \alpha\, y^2) $$

Com $\alpha = 50$, é uma **bacia muito alongada**: 50× mais inclinada na direção `y` do que na direção `x`. O mínimo é trivialmente $(0, 0)$.

GD com `lr` único tem que escolher entre:
- **`lr` pequeno** — não diverge na direção íngreme (`y`), mas avança lentamente na suave (`x`).
- **`lr` grande** — diverge na direção íngreme.

**Resultado:** zigzag perpendicular ao vale. Vais ver com os teus olhos.

### Task 1.1 — GD num vale alongado

Implementa `f(x)` e `grad_f(x)` para $f(x,y) = (x^2 + 50 y^2)/2$ aceitando `x` como vetor 2D.

**Gradiente:** $\nabla f(x, y) = (x,\ 50 y)$.

Depois implementa um GD vetorial genérico `gd(x0, grad_fn, lr, n_iter)` que devolve a história (lista de vetores).

In [ ]:
ALPHA = 50.0

def f_valley(x):
    """f(x, y) = 0.5 * (x² + α y²). x deve ser array 1D de tamanho 2."""
    x = np.asarray(x, dtype=float)
    # TODO: devolve 0.5 * (x[0]**2 + ALPHA * x[1]**2)
    return ...

def grad_valley(x):
    """∇f = (x, α y)."""
    x = np.asarray(x, dtype=float)
    # TODO: devolve np.array([x[0], ALPHA * x[1]])
    return ...

def gd(x0, grad_fn, lr, n_iter):
    """GD genérico. Devolve numpy array (n_iter+1, dim)."""
    x = np.asarray(x0, dtype=float).copy()
    history = [x.copy()]
    for _ in range(n_iter):
        # TODO: x = x - lr * grad_fn(x)
        ...
        history.append(x.copy())
    return np.array(history)

# Sanity em (0,0): gradiente nulo
check(np.allclose(grad_valley([0.0, 0.0]), [0.0, 0.0]), 'gradiente em (0,0) deve ser zero')

# GD com lr seguro: deve convergir mas devagar (a direção x é "achatada", lr efetivo=lr)
hist = gd([5.0, 1.0], grad_valley, lr=0.02, n_iter=200)
final = hist[-1]
check(np.linalg.norm(final) < 1.0, f'após 200 iter com lr=0.02, ||x|| < 1, obteve {np.linalg.norm(final):.3f}')

# A iteração na direção y é y_{t+1} = (1 - lr·α) y_t.
# Para convergir, |1 - lr·α| < 1  ⟹  0 < lr < 2/α = 0.04.
# lr = 2/α: y oscila com módulo constante (factor = -1).
# lr > 2/α: y diverge.
hist_div = gd([5.0, 1.0], grad_valley, lr=0.05, n_iter=50)
check(abs(hist_div[-1, 1]) > abs(hist_div[0, 1]),
      'lr=0.05 > 2/α=0.04 deve fazer y divergir')

# lr = 2/α exato: módulo de y mantém-se (orbita sem amortecer)
hist_edge = gd([5.0, 1.0], grad_valley, lr=2.0/ALPHA, n_iter=50)
check(abs(abs(hist_edge[-1, 1]) - 1.0) < 1e-9,
      f'em lr=2/α, |y| mantém-se em 1, obteve |y|={abs(hist_edge[-1, 1])}')

### Task 1.2 — Visualizar o zigzag

Plota a trajetória de GD com `lr=0.038` (perto do limite) sobre as curvas de nível de $f$. Vais ver o caminho a oscilar perpendicularmente ao vale — *este é o problema que o momentum vai resolver*.

In [ ]:
# Curvas de nível
xs = np.linspace(-6, 6, 200)
ys = np.linspace(-1.5, 1.5, 200)
X, Y = np.meshgrid(xs, ys)
Z = 0.5 * (X**2 + ALPHA * Y**2)

hist_gd = gd([5.0, 1.0], grad_valley, lr=0.038, n_iter=80)

fig, ax = plt.subplots(figsize=(9, 4))
ax.contour(X, Y, Z, levels=np.logspace(-1, 2.5, 12), cmap='gray')
ax.plot(hist_gd[:, 0], hist_gd[:, 1], 'b.-', markersize=4, alpha=0.7, label='GD (lr=0.038)')
ax.plot(0, 0, 'r*', markersize=15, label='mínimo')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('GD num vale alongado: o zigzag')
ax.legend(); ax.set_aspect('equal')
plt.show()

# Quantifica o zigzag: as componentes y devem alternar sinal frequentemente
ys_traj = hist_gd[:, 1]
sign_changes = sum(1 for i in range(1, len(ys_traj)) if ys_traj[i] * ys_traj[i-1] < 0)
print(f'mudanças de sinal em y: {sign_changes} (de {len(ys_traj)-1} passos)')
check(sign_changes > 30, f'GD perto do limite deve oscilar muito (>30 sign changes), obteve {sign_changes}')

## §2 — Momentum (heavy ball)

Polyak (1964). A ideia: a *velocidade* não é o gradiente atual; é uma média exponencial dos gradientes passados.

$$ v_{t+1} = \beta\, v_t + \nabla f(x_t) \\ x_{t+1} = x_t - \eta\, v_{t+1} $$

`β` é o coeficiente de momento (tipicamente 0.9). `v_0 = 0`.

**Intuição:** se os gradientes recentes apontam todos na mesma direção (a direção do vale, suave), `v` cresce e dá-te velocidade ali. Se oscilarem (a direção íngreme do zigzag), cancelam-se. Resultado: amortece o zigzag e acelera no vale.

### Task 2.1 — Implementa GD com momento e compara

In [ ]:
def gd_momentum(x0, grad_fn, lr, beta, n_iter):
    """GD com momento (heavy ball)."""
    x = np.asarray(x0, dtype=float).copy()
    v = np.zeros_like(x)
    history = [x.copy()]
    for _ in range(n_iter):
        # TODO: v = beta * v + grad_fn(x)
        # TODO: x = x - lr * v
        ...
        history.append(x.copy())
    return np.array(history)

# Comparação: 3 corridas a 80 iterações.
#   GD lr=0.038  (perto do limite, zigzag observado em §1.2)
#   GD lr=0.01   (lr "seguro" mas lento na direção x)
#   Momentum lr=0.01, β=0.9 — acumula velocidade, deve bater AMBOS os GDs
hist_gd_slow = gd([5.0, 1.0], grad_valley, lr=0.01, n_iter=80)
hist_mom = gd_momentum([5.0, 1.0], grad_valley, lr=0.01, beta=0.9, n_iter=80)

final_gd_zigzag = np.linalg.norm(hist_gd[-1])
final_gd_slow = np.linalg.norm(hist_gd_slow[-1])
final_mom = np.linalg.norm(hist_mom[-1])
print(f'||final|| GD lr=0.038 (zigzag): {final_gd_zigzag:.4f}')
print(f'||final|| GD lr=0.01 (lento)  : {final_gd_slow:.4f}')
print(f'||final|| Momentum lr=0.01    : {final_mom:.4f}')
check(final_mom < final_gd_slow,
      'momentum deve bater GD à mesma lr=0.01 (acelera na direção do vale)')

# Visualiza ambas trajetórias
fig, ax = plt.subplots(figsize=(9, 4))
ax.contour(X, Y, Z, levels=np.logspace(-1, 2.5, 12), cmap='gray')
ax.plot(hist_gd[:, 0], hist_gd[:, 1], 'b.-', markersize=3, alpha=0.5, label='GD lr=0.038')
ax.plot(hist_mom[:, 0], hist_mom[:, 1], 'r.-', markersize=3, alpha=0.7, label='Momentum lr=0.01, β=0.9')
ax.plot(0, 0, 'k*', markersize=15)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(); ax.set_aspect('equal')
ax.set_title('GD vs Momentum no vale alongado')
plt.show()

## §3 — Nesterov Accelerated Gradient

Nesterov (1983). Pequena modificação ao momentum, melhoria teórica significativa.

A ideia: em vez de avaliar o gradiente em $x_t$, primeiro **dá um passo de momentum** (lookahead) e avalia o gradiente lá.

$$ x_{\text{look}} = x_t - \eta\, \beta\, v_t \\ v_{t+1} = \beta\, v_t + \nabla f(x_{\text{look}}) \\ x_{t+1} = x_t - \eta\, v_{t+1} $$

Para funções convexas com gradiente Lipschitz, Nesterov atinge taxa de convergência $O(1/k^2)$ — o limite teórico para métodos de primeira ordem (Boyd Cap 9 explica porquê).

### Task 3.1 — Implementa NAG

In [ ]:
def gd_nesterov(x0, grad_fn, lr, beta, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    v = np.zeros_like(x)
    history = [x.copy()]
    for _ in range(n_iter):
        # TODO: lookahead = x - lr * beta * v
        # TODO: g = grad_fn(lookahead)
        # TODO: v = beta * v + g
        # TODO: x = x - lr * v
        ...
        history.append(x.copy())
    return np.array(history)

hist_nag = gd_nesterov([5.0, 1.0], grad_valley, lr=0.01, beta=0.9, n_iter=80)

# NAG deve convergir pelo menos tão bem como momentum (com mesma config)
final_nag = np.linalg.norm(hist_nag[-1])
print(f'||final|| Momentum lr=0.01, β=0.9 : {final_mom:.4f}')
print(f'||final|| Nesterov lr=0.01, β=0.9 : {final_nag:.4f}')
check(final_nag <= final_mom * 1.5,
      'NAG deve ser comparável ou melhor que momentum à mesma config')
check(final_nag < final_gd_slow,
      'NAG deve bater GD lr=0.01 (à mesma lr, com acelaração)')

# Aviso pedagógico: NAG/Momentum com β alto (0.9) e lr alto (0.038) DIVERGEM neste
# vale ill-conditioned. Estabilidade: lr < 2(1+β)/λ_max = 2·1.9/50 ≈ 0.076 para momentum;
# NAG é ainda mais sensível. Se subires lr para 0.038 + β=0.9, vês a divergência.

## §4 — AdaGrad: passos diferentes para cada parâmetro

Duchi et al., 2011. A intuição: numa direção em que o gradiente é grande (íngreme), queres passos pequenos. Numa direção em que o gradiente é pequeno (suave), queres passos grandes.

AdaGrad acumula o quadrado de cada componente do gradiente e divide o passo por essa raiz:

$$ G_{t+1} = G_t + g_t \odot g_t \\ x_{t+1} = x_t - \frac{\eta}{\sqrt{G_{t+1}} + \epsilon} \odot g_t $$

Onde $\odot$ é multiplicação elemento-a-elemento e $\epsilon \approx 10^{-8}$ evita divisão por zero. `G` começa em zero.

**Trade-off:** $G$ só cresce → eventualmente os passos morrem. Adam (CP04) resolve isso com média móvel.

### Task 4.1 — Implementa AdaGrad

In [ ]:
def gd_adagrad(x0, grad_fn, lr, n_iter, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    G = np.zeros_like(x)  # acumulador de gradientes ao quadrado
    history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(x)
        # TODO: G = G + g * g
        # TODO: x = x - lr * g / (np.sqrt(G) + eps)
        ...
        history.append(x.copy())
    return np.array(history)

# AdaGrad pode usar uma lr global maior porque cada componente é escalada
hist_ada = gd_adagrad([5.0, 1.0], grad_valley, lr=1.0, n_iter=80)

final_ada_norm = np.linalg.norm(hist_ada[-1])
print(f'||final|| AdaGrad = {final_ada_norm:.4f}')
check(final_ada_norm < 1.0, f'AdaGrad deve convergir, obteve {final_ada_norm:.4f}')

# AdaGrad deve mostrar trajetória mais "estável" (sem zigzag agressivo)
ys_ada = hist_ada[:, 1]
sign_changes_ada = sum(1 for i in range(1, len(ys_ada)) if ys_ada[i] * ys_ada[i-1] < 0)
print(f'AdaGrad sign changes em y: {sign_changes_ada} (vs GD puro {sign_changes})')
check(sign_changes_ada < sign_changes,
      f'AdaGrad deve oscilar menos que GD puro: {sign_changes_ada} vs {sign_changes}')

## §5 — Aplicação: regressão linear via GD

Pega no dataset sintético do CP02 ($y = 2x + 1 + \text{ruído}$) e ajusta os parâmetros $w, b$ minimizando MSE com cada um dos 4 métodos. Compara as curvas de convergência.

A perda e o gradiente são:

$$ L(w, b) = \frac{1}{n} \sum_i (w x_i + b - y_i)^2 \\ \frac{\partial L}{\partial w} = \frac{2}{n} \sum_i x_i (w x_i + b - y_i),\quad \frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (w x_i + b - y_i) $$

### Task 5.1 — Gradiente do MSE e comparação dos 4 métodos

In [ ]:
# Mesmo dataset do CP02
n_pts = 30
x_data = rng.uniform(-3, 3, size=n_pts)
y_data = 2.0 * x_data + 1.0 + rng.normal(scale=0.3, size=n_pts)

def loss_wb(params):
    w, b = params
    err = w * x_data + b - y_data
    return float(np.mean(err ** 2))

def grad_wb(params):
    """Gradiente de MSE w.r.t. (w, b)."""
    w, b = params
    err = w * x_data + b - y_data
    # TODO: dL/dw = (2/n) * sum(x_i * err_i)
    # TODO: dL/db = (2/n) * sum(err_i)
    # TODO: devolve np.array([dL_dw, dL_db])
    return ...

# Sanity: gradiente em (w=2, b=1) deve ser pequeno (perto da solução)
g_at_truth = grad_wb([2.0, 1.0])
check(np.linalg.norm(g_at_truth) < 0.5,
      f'gradiente perto da verdade deve ser pequeno, obteve ||g||={np.linalg.norm(g_at_truth):.3f}')

# Corre 4 otimizadores a partir de (w=0, b=0), 200 iterações
n_iter = 200
x0 = np.array([0.0, 0.0])

hist_gd_lin = gd(x0, grad_wb, lr=0.05, n_iter=n_iter)
hist_mom_lin = gd_momentum(x0, grad_wb, lr=0.05, beta=0.9, n_iter=n_iter)
hist_nag_lin = gd_nesterov(x0, grad_wb, lr=0.05, beta=0.9, n_iter=n_iter)
hist_ada_lin = gd_adagrad(x0, grad_wb, lr=0.5, n_iter=n_iter)

# Calcula a perda em cada iteração
def losses(hist):
    return np.array([loss_wb(p) for p in hist])

L_gd = losses(hist_gd_lin)
L_mom = losses(hist_mom_lin)
L_nag = losses(hist_nag_lin)
L_ada = losses(hist_ada_lin)

print(f'L_final GD     : {L_gd[-1]:.6f}')
print(f'L_final Mom    : {L_mom[-1]:.6f}')
print(f'L_final NAG    : {L_nag[-1]:.6f}')
print(f'L_final AdaGrad: {L_ada[-1]:.6f}')

# Todos devem convergir (perda final pequena)
for name, L in [('GD', L_gd), ('Mom', L_mom), ('NAG', L_nag), ('AdaGrad', L_ada)]:
    check(L[-1] < 0.5, f'{name} deve atingir perda < 0.5, obteve {L[-1]:.4f}')

# Pelo menos um dos accelerated (mom/NAG) deve ser melhor que GD puro às mesmas 200 iter
check(min(L_mom[-1], L_nag[-1]) <= L_gd[-1] + 1e-6,
      f'momentum ou Nesterov deve bater GD: GD={L_gd[-1]:.5f}, melhor acelerado={min(L_mom[-1], L_nag[-1]):.5f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(L_gd, label='GD')
ax.semilogy(L_mom, label='Momentum')
ax.semilogy(L_nag, label='Nesterov')
ax.semilogy(L_ada, label='AdaGrad')
ax.set_xlabel('iteração'); ax.set_ylabel('MSE (log)')
ax.set_title('4 otimizadores em least squares')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()

## 🏁 Velocity — fechado

Se chegaste aqui sem `AssertionError`, dominaste:

1. **O problema** — viste com os teus olhos porque GD puro zigzaga em vales alongados.
2. **Momentum** (Polyak) — média exponencial de gradientes; amortece o zigzag.
3. **Nesterov** — lookahead antes do passo; convergência $O(1/k^2)$ no convexo.
4. **AdaGrad** — passos por-coordenada; adapta-se à curvatura local sem precisares de Hessiana.
5. **Aplicação** — regressão linear via 4 métodos; curvas de convergência comparadas.

**Perguntas de fecho:**
- Por que é que momentum precisa que `β < 1`?
- O que significa "gradiente Lipschitz"? Onde aparece o $L$ (constante de Lipschitz) na escolha de `lr`?
- Por que é que AdaGrad **morre** em problemas longos? (Pista: $G$ é monotonamente crescente.)

**Próximo checkpoint:** [04 — Newton](../04_Checkpoint_Newton/) — saímos das primeiras derivadas. Hessiana, método de Newton 2D, e os otimizadores adaptativos modernos (RMSProp, Adam, AdamW).